In [1]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import re
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
from pymystem3 import Mystem
import warnings
warnings.filterwarnings('ignore')

# Скачаем стоп-слова, если их нет
try:
    stopwords.words('russian')
except:
    nltk.download('stopwords')
    
print("Библиотеки импортированы успешно")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/dmitryvokhmin/nltk_data...


Библиотеки импортированы успешно


[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
# Загрузка данных
products_df = pd.read_csv('data/ProductsDataset.csv')
print(f"Загружено товаров: {len(products_df)}")
print(f"Колонки: {products_df.columns.tolist()}")
products_df.head()

Загружено товаров: 35548
Колонки: ['title', 'descrirption', 'product_id', 'category_id', 'subcategory_id', 'properties', 'image_links']


,title,descrirption,product_id,category_id,subcategory_id,properties,image_links
0,Юбка детская ORBY,"Новая, не носили ни разу. В реале красивей чем...",58e3cfe6132ca50e053f5f82,22.0,2211,"{'detskie_razmer_rost': '81-86 (1,5 года)'}",http://cache3.youla.io/files/images/360_360/58...
1,Ботильоны,"Новые,привезены из Чехии ,указан размер 40,но ...",5667531b2b7f8d127d838c34,9.0,902,"{'zhenskaya_odezhda_tzvet': 'Зеленый', 'visota...",http://cache3.youla.io/files/images/360_360/5b...
2,Брюки,Размер 40-42. Брюки почти новые - не знаю как ...,59534826aaab284cba337e06,9.0,906,{'zhenskaya_odezhda_dzhinsy_bryuki_tip': 'Брюк...,http://cache3.youla.io/files/images/360_360/59...
3,Продам детские шапки,"Продам шапки,кажда 200р.Розовая и белая проданны.",57de544096ad842e26de8027,22.0,2217,"{'detskie_pol': 'Девочкам', 'detskaya_odezhda_...",http://cache3.youla.io/files/images/360_360/57...
4,Блузка,"Темно-синяя, 42 размер,состояние отличное,как ...",5ad4d2626c86cb168d212022,9.0,907,"{'zhenskaya_odezhda_tzvet': 'Синий', 'zhenskay...",http://cache3.youla.io/files/images/360_360/5a...


In [3]:
# Загрузка данных для болталки
def load_qa_data(filepath):
    """Загрузка вопросов и ответов из файла"""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    qa_pairs = []
    blocks = content.split('---')[1:]  # Разделяем по ---
    
    for block in blocks:
        lines = [l.strip() for l in block.strip().split('\n') if l.strip()]
        if len(lines) >= 2:
            question = lines[0].rstrip('. ')
            # Берем первый ответ (или можем выбрать лучший)
            answer = lines[1] if len(lines) > 1 else "Не знаю"
            qa_pairs.append({'question': question, 'answer': answer})
    
    return pd.DataFrame(qa_pairs)

qa_df = load_qa_data('data/prepared_answers.txt')
print(f"Загружено вопросов для болталки: {len(qa_df)}")
qa_df.head()

Загружено вопросов для болталки: 7894


,question,answer
0,) \было одето \ (сказ.= ) платье (подлеж. -)...,переехала в москву хочу общаться и тусить! С К...
1,\<br>снизу объеденить фигурной скобкой и напис...,что можно сделать из дерева в 6 калассе .\tБур...
2,--<br>Торт «Черепаха» <br>мука - 2 стакана <br...,Зачем Мужчины)) ставят фото котят на свои авки...
3,"<br>наверное есть ещё какие-нить коды, - но с ...",Ноутбук Core i7-4710HQ 2.5ГГц/GeForce GTX860M ...
4,веселиться он так!!!!,Меня замучил этот стажер! куда бы его послать?...


In [4]:
# Функция препроцессинга текста
mystem = Mystem()
russian_stopwords = set(stopwords.words('russian'))

def preprocess_text(text):
    """
    Препроцессинг текста:
    - Приведение к нижнему регистру
    - Удаление знаков препинания
    - Лемматизация
    - Удаление стоп-слов
    """
    if pd.isna(text):
        return ""
    
    # Приведение к нижнему регистру
    text = str(text).lower()
    
    # Удаление знаков препинания и спецсимволов
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Лемматизация
    lemmas = mystem.lemmatize(text)
    
    # Удаление стоп-слов и коротких слов
    words = [word for word in lemmas if word.strip() and word not in russian_stopwords and len(word) > 2]
    
    return ' '.join(words)

# Тестирование
test_text = "Юбка детская ORBY"
print(f"Оригинал: {test_text}")
print(f"После обработки: {preprocess_text(test_text)}")

Installing mystem to /Users/dmitryvokhmin/.local/bin/mystem from http://download.cdn.yandex.net/mystem/mystem-3.1-macosx.tar.gz


Оригинал: Юбка детская ORBY
После обработки: юбка детский orby


In [5]:
# Создание датасета для классификатора
# Товары - это класс 1, болталка - класс 0

# Подготовка товаров (класс 1)
products_df['text'] = products_df['title'] + ' ' + products_df['descrirption'].fillna('')
products_df['text_processed'] = products_df['text'].apply(preprocess_text)
products_df['label'] = 1

# Подготовка вопросов для болталки (класс 0)
qa_df['text_processed'] = qa_df['question'].apply(preprocess_text)
qa_df['label'] = 0

# Объединяем датасеты
product_texts = products_df[['text_processed', 'label']].copy()
qa_texts = qa_df[['text_processed', 'label']].copy()

# Берем все товары и сбалансируем количество вопросов
n_samples = min(len(product_texts), len(qa_texts))
print(f"Товаров: {len(product_texts)}, Вопросов: {len(qa_texts)}")

# Создаем сбалансированный датасет
if len(qa_texts) > len(product_texts):
    qa_texts = qa_texts.sample(n=len(product_texts), random_state=42)
    
classifier_data = pd.concat([product_texts, qa_texts], ignore_index=True)
classifier_data = classifier_data[classifier_data['text_processed'].str.len() > 0]  # Убираем пустые
classifier_data = classifier_data.sample(frac=1, random_state=42).reset_index(drop=True)  # Перемешиваем

print(f"\\nРазмер итогового датасета: {len(classifier_data)}")
print(f"Распределение классов:\\n{classifier_data['label'].value_counts()}")
classifier_data.head(10)

Товаров: 35548, Вопросов: 7894
\nРазмер итогового датасета: 42243
Распределение классов:\nlabel
1    35544
0     6699
Name: count, dtype: int64


,text_processed,label
0,ехать пристегувшись ремень новый кольцевой нах...,0
1,cacl моль ____ моль __________ моль моль мо...,0
2,платье tara jarmon продавать платье tara jarmo...,1
3,это самый сильный стихотворение любовь любимый,0
4,молокоотсос medela медеть предлагать ваш цена ...,1
5,туфля продавать туфля нат кожа,1
6,платье девочка платье год новый сарафан puledr...,1
7,костюм мужской мужской строгий костюм,1
8,сарафан вязаный девочка новый ручной работа вя...,1
9,оригинальный мягкий кукла хороший состояние,1


In [6]:
# Разделение на обучающую и валидационную выборки
X = classifier_data['text_processed']
y = classifier_data['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Обучающая выборка: {len(X_train)}")
print(f"Валидационная выборка: {len(X_val)}")
print(f"\\nРаспределение в train:\\n{y_train.value_counts()}")
print(f"\\nРаспределение в val:\\n{y_val.value_counts()}")

Обучающая выборка: 33794
Валидационная выборка: 8449
\nРаспределение в train:\nlabel
1    28435
0     5359
Name: count, dtype: int64
\nРаспределение в val:\nlabel
1    7109
0    1340
Name: count, dtype: int64


In [7]:
# Векторизация с помощью TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

print(f"Размерность векторов: {X_train_vec.shape[1]}")
print(f"Размер обучающей матрицы: {X_train_vec.shape}")
print(f"Размер валидационной матрицы: {X_val_vec.shape}")

Размерность векторов: 5000
Размер обучающей матрицы: (33794, 5000)
Размер валидационной матрицы: (8449, 5000)


In [8]:
# Обучение логистической регрессии
classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X_train_vec, y_train)

# Предсказания на валидации
y_pred = classifier.predict(X_val_vec)

# Метрики
accuracy = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print(f"Accuracy на валидации: {accuracy:.4f}")
print(f"F1-score на валидации: {f1:.4f}")
print(f"\\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=['Болталка', 'Товар']))

Accuracy на валидации: 0.9865
F1-score на валидации: 0.9920
\nClassification Report:
              precision    recall  f1-score   support

    Болталка       0.96      0.96      0.96      1340
       Товар       0.99      0.99      0.99      7109

    accuracy                           0.99      8449
   macro avg       0.97      0.97      0.97      8449
weighted avg       0.99      0.99      0.99      8449



In [9]:
# Сохранение модели классификатора
with open('models/classifier.pkl', 'wb') as f:
    pickle.dump(classifier, f)
    
with open('models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
    
print("Модель и векторизатор сохранены")

Модель и векторизатор сохранены


In [10]:
# Подготовка данных для Word2Vec (токенизация)
def tokenize(text):
    """Токенизация текста для Word2Vec"""
    return text.split()

# Подготовка товаров
products_df['tokens'] = products_df['text_processed'].apply(tokenize)
product_tokens = products_df['tokens'].tolist()

# Подготовка вопросов
qa_df['tokens'] = qa_df['text_processed'].apply(tokenize)
qa_tokens = qa_df['tokens'].tolist()

print(f"Подготовлено товаров для Word2Vec: {len(product_tokens)}")
print(f"Подготовлено вопросов для Word2Vec: {len(qa_tokens)}")
print(f"\\nПример токенов товара: {product_tokens[0]}")
print(f"Пример токенов вопроса: {qa_tokens[0]}")

Подготовлено товаров для Word2Vec: 35548
Подготовлено вопросов для Word2Vec: 7894
\nПример токенов товара: ['юбка', 'детский', 'orby', 'новый', 'носить', 'реал', 'красиво', 'фото']
Пример токенов вопроса: ['одевать', 'сказ', 'платье', 'подлеж', 'марля', 'опреливать', 'раскалять', 'голубой', 'розовый', 'желтый', 'краска', 'опреление', 'выраж', 'причастный', 'оборот']


In [11]:
# Обучение Word2Vec модели
all_tokens = product_tokens + qa_tokens

w2v_model = Word2Vec(
    sentences=all_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    seed=42
)

print(f"Модель Word2Vec обучена")
print(f"Словарь содержит {len(w2v_model.wv)} слов")
print(f"Размерность векторов: {w2v_model.wv.vector_size}")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Модель Word2Vec обучена
Словарь содержит 22683 слов
Размерность векторов: 100


In [12]:
# Функция для преобразования текста в вектор Word2Vec
def text_to_vector(tokens, model):
    """Преобразует токенизированный текст в усредненный вектор Word2Vec"""
    vectors = []
    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])
    
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.wv.vector_size)

# Создание векторов для всех товаров
products_df['vector'] = products_df['tokens'].apply(lambda x: text_to_vector(x, w2v_model))
product_vectors = np.vstack(products_df['vector'].values)

print(f"Создано {product_vectors.shape[0]} векторов для товаров")
print(f"Размерность каждого вектора: {product_vectors.shape[1]}")

Создано 35548 векторов для товаров
Размерность каждого вектора: 100


In [13]:
# Создание векторов для вопросов болталки
qa_df['vector'] = qa_df['tokens'].apply(lambda x: text_to_vector(x, w2v_model))
qa_vectors = np.vstack(qa_df['vector'].values)

print(f"Создано {qa_vectors.shape[0]} векторов для вопросов")
print(f"Размерность каждого вектора: {qa_vectors.shape[1]}")

Создано 7894 векторов для вопросов
Размерность каждого вектора: 100


In [14]:
# Построение индексов для быстрого поиска
from sklearn.metrics.pairwise import cosine_similarity

def find_most_similar(query_vector, vectors, top_k=1):
    """Находит наиболее похожие векторы по косинусному сходству"""
    similarities = cosine_similarity([query_vector], vectors)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return top_indices, similarities[top_indices]

# Тестирование поиска товара
test_query = "юбка детская orby"
test_tokens = tokenize(preprocess_text(test_query))
test_vector = text_to_vector(test_tokens, w2v_model)

indices, scores = find_most_similar(test_vector, product_vectors, top_k=3)
print(f"Запрос: {test_query}")
print(f"\\nТоп-3 похожих товара:")
for i, (idx, score) in enumerate(zip(indices, scores), 1):
    print(f"{i}. [{score:.4f}] {products_df.iloc[idx]['title']} (ID: {products_df.iloc[idx]['product_id']})")

Запрос: юбка детская orby
\nТоп-3 похожих товара:
1. [0.9612] Юбка детская (ID: 5b07bceb22a44943045a0a62)
2. [0.9132] Юбка детская (ID: 5ad469cec15ae36576339ca0)
3. [0.8852] плюшевая юбка (ID: 587a55f3c5c2e6ef183b873d)


In [30]:
# Сохранение данных для поиска
w2v_model.save('models/word2vec.model')

# Создаем TF-IDF векторизатор для поиска товаров
product_search_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
product_tfidf_matrix = product_search_vectorizer.fit_transform(products_df['text_processed'])

# Сохранение индексов и данных
search_data = {
    'product_vectors': product_vectors,
    'product_ids': products_df['product_id'].tolist(),
    'product_titles': products_df['title'].tolist(),
    'product_texts': products_df['text_processed'].tolist(),
    'product_tfidf_matrix': product_tfidf_matrix,
    'qa_vectors': qa_vectors,
    'qa_questions': qa_df['question'].tolist(),
    'qa_answers': qa_df['answer'].tolist()
}

with open('models/search_data.pkl', 'wb') as f:
    pickle.dump(search_data, f)

with open('models/product_search_vectorizer.pkl', 'wb') as f:
    pickle.dump(product_search_vectorizer, f)

print("Word2Vec модель и данные для поиска сохранены")

Word2Vec модель и данные для поиска сохранены


In [31]:
# Реализация функции get_answer()

class ChatBot:
    def __init__(self):
        """Инициализация чат-бота"""
        self.classifier = None
        self.vectorizer = None
        self.w2v_model = None
        self.search_data = None
        self.product_search_vectorizer = None
        self.mystem = Mystem()
        self.stopwords = set(stopwords.words('russian'))
        
    def load_models(self, models_dir='models'):
        """Загрузка всех необходимых моделей"""
        with open(f'{models_dir}/classifier.pkl', 'rb') as f:
            self.classifier = pickle.load(f)
        
        with open(f'{models_dir}/vectorizer.pkl', 'rb') as f:
            self.vectorizer = pickle.load(f)
        
        self.w2v_model = Word2Vec.load(f'{models_dir}/word2vec.model')
        
        with open(f'{models_dir}/search_data.pkl', 'rb') as f:
            self.search_data = pickle.load(f)
        
        with open(f'{models_dir}/product_search_vectorizer.pkl', 'rb') as f:
            self.product_search_vectorizer = pickle.load(f)
            
        print("Все модели загружены успешно")
    
    def preprocess_text(self, text):
        """Препроцессинг текста"""
        if pd.isna(text):
            return ""
        
        text = str(text).lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\d+', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        
        lemmas = self.mystem.lemmatize(text)
        words = [word for word in lemmas if word.strip() and word not in self.stopwords and len(word) > 2]
        
        return ' '.join(words)
    
    def tokenize(self, text):
        """Токенизация"""
        return text.split()
    
    def text_to_vector(self, tokens):
        """Преобразование токенов в вектор Word2Vec"""
        vectors = []
        for word in tokens:
            if word in self.w2v_model.wv:
                vectors.append(self.w2v_model.wv[word])
        
        if len(vectors) > 0:
            return np.mean(vectors, axis=0)
        else:
            return np.zeros(self.w2v_model.wv.vector_size)
    
    def find_most_similar(self, query_vector, vectors, top_k=1):
        """Поиск наиболее похожих векторов"""
        similarities = cosine_similarity([query_vector], vectors)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return top_indices, similarities[top_indices]
    
    def get_answer(self, query):
        """
        Основной метод для получения ответа на запрос
        
        Args:
            query: текстовый запрос пользователя
            
        Returns:
            Ответ в виде строки
        """
        processed_query = self.preprocess_text(query)
        
        # Классификация: товарный запрос или болталка
        query_vec = self.vectorizer.transform([processed_query])
        is_product = self.classifier.predict(query_vec)[0]
        
        if is_product == 1:
            # Товарный запрос - используем TF-IDF для точного поиска
            query_tfidf = self.product_search_vectorizer.transform([processed_query])
            similarities = cosine_similarity(query_tfidf, self.search_data['product_tfidf_matrix'])[0]
            best_idx = np.argmax(similarities)
            
            product_id = self.search_data['product_ids'][best_idx]
            product_title = self.search_data['product_titles'][best_idx]
            
            return f"{product_id} {product_title}"
        else:
            # Болталка - используем Word2Vec для семантического поиска
            tokens = self.tokenize(processed_query)
            query_vector = self.text_to_vector(tokens)
            
            indices, scores = self.find_most_similar(
                query_vector,
                self.search_data['qa_vectors'],
                top_k=1
            )
            
            answer = self.search_data['qa_answers'][indices[0]]
            return answer

# Создание экземпляра бота
bot = ChatBot()
print("ChatBot создан")

ChatBot создан


In [32]:
# Создаем директорию для моделей, если её нет
import os
os.makedirs('models', exist_ok=True)
print("Директория models создана")

Директория models создана


In [33]:
# Создание глобальной функции get_answer для тестов
def get_answer(query):
    """Глобальная функция для автотестов"""
    return bot.get_answer(query)

print("Функция get_answer() создана")

Функция get_answer() создана


In [34]:
# Автотесты
print("=== АВТОТЕСТЫ ===\\n")
bot.load_models()

# Тест 1: Товарный запрос
test1_query = "Юбка детская ORBY"
test1_result = get_answer(test1_query)
print(f"Тест 1: {test1_query}")
print(f"Ответ: {test1_result}")
print(f"Проверка: startswith('58e3cfe6132ca50e053f5f82') = {test1_result.startswith('58e3cfe6132ca50e053f5f82')}")
assert test1_result.startswith("58e3cfe6132ca50e053f5f82"), f"Тест 1 не прошел! Ожидалось, что ответ начинается с '58e3cfe6132ca50e053f5f82', получено: {test1_result}"
print("✓ Тест 1 ПРОЙДЕН\\n")

# Тест 2: Болталка (не товарный запрос)
test2_query = "Где ключи от танка"
test2_result = get_answer(test2_query)
print(f"Тест 2: {test2_query}")
print(f"Ответ: {test2_result}")
print(f"Проверка: not startswith('5') = {not test2_result.startswith('5')}")
assert not test2_result.startswith("5"), f"Тест 2 не прошел! Ответ не должен начинаться с '5', получено: {test2_result}"
print("✓ Тест 2 ПРОЙДЕН\\n")

print("=== ВСЕ АВТОТЕСТЫ ПРОЙДЕНЫ УСПЕШНО ===")

=== АВТОТЕСТЫ ===\n
Все модели загружены успешно
Тест 1: Юбка детская ORBY
Ответ: 58e3cfe6132ca50e053f5f82 Юбка детская ORBY
Проверка: startswith('58e3cfe6132ca50e053f5f82') = True
✓ Тест 1 ПРОЙДЕН\n
Тест 2: Где ключи от танка
Ответ: Даны вещества:NaBr,K2SO4,H2SO4,AICI3,AI2O3,ZnS,ZnCL2,K2O,NaOH,Ba(NO3)2. Используя толко эти вещества,получите ? А)сульфат калия Б)сульфат амолиния В)гидроксид цинка.	<p> ) гидроксид цинка это есть в  домашке у меня<br></p>.
Проверка: not startswith('5') = True
✓ Тест 2 ПРОЙДЕН\n
=== ВСЕ АВТОТЕСТЫ ПРОЙДЕНЫ УСПЕШНО ===
